# Stage 1 &mdash; baseline multi-label CUI classifier for ROCOv2

Notebook counterpart of `roco_cui_classifier.py`. Runs on **CUDA, Apple Silicon (MPS),
or CPU** &mdash; auto-detected.

### Why multi-LABEL, not multi-class
ROCOv2 images carry **3.35 concepts on average** (max 21). Concepts must therefore be
predicted **independently**: one sigmoid per class + BCE. Softmax + cross-entropy would
force exactly one label per image and could not represent
`{CT, Chest, Pleural effusion}` simultaneously.

### Architecture, and why
Justified by the ImageCLEFmedical 2025 concept-detection leaderboard, where
ImageNet-pretrained CNNs beat both a medical transformer (MedCLIP, 0.4003) and an
8B VLM (LLaVA-LLaMA-3, 0.3982):

| Rank | Approach | F1 |
|---|---|---|
| 1 | EfficientNet-B0 + **DenseNet-121** + ConvNeXt-Tiny | 0.5888 |
| 2 | EfficientNet-B0 + **DenseNet-121** | 0.5766 |

DenseNet-121 appears in **both** top-2 systems, so it is our single-model starting point.

```
  DenseNet-121 features (ImageNet-pretrained)
        -> GeM pooling  (learnable p; interpolates average <-> max)
        -> Linear(1024 -> 1571)
        -> sigmoid  =>  1571 independent probabilities
```

**GeM pooling** matters because concepts live at different scales: "CT scan" is a global
property of the whole image, while "pleural effusion" is a small local region. Average
pooling washes out the small finding; max pooling discards context. GeM learns where to sit.

### Primary metric
**Samples-average F1** &mdash; per-image F1 between predicted and gold concept sets,
averaged over images. This is ImageCLEF's official metric, so our number is directly
comparable (AUEB won 2025 with **0.5888**).

In [ ]:
import os, json, math, time, random, collections
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print("device =", device)

In [ ]:
# ------------------------------- Config ------------------------------- #
ROCO_DIR   = "./rocov2"        # <-- point at your local copy of the dataset
OUT_DIR    = "./cui_clf_ckpt"
MIN_FREQ   = 10                # ImageCLEF's own curation rule -> 1571 concepts
IMG_SIZE   = 224
BATCH      = 32
EPOCHS     = 15
LR         = 1e-4
WEIGHT_DEC = 1e-4
POSW_CLAMP = 50.0
N_TRAIN    = 0                 # 0 = all 59,958;  set e.g. 2000 for a quick smoke test
N_VAL      = 0                 # 0 = all 9,904
WORKERS    = 0                 # 0 = load in this process.  KEEP 0 IN A NOTEBOOK on
                               # macOS/Windows: those spawn worker processes, and a
                               # spawned worker re-imports "__main__" -- which for a
                               # kernel is the ipykernel launcher, not this notebook,
                               # so RocoCUI cannot be found and the loader dies.
                               # On Linux/Colab (fork) 4 is safe; the .py script runs
                               # workers on any OS because it guards main().
SEED       = 42
PATIENCE   = 4

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# --------- Label space, built from TRAIN ONLY (never valid/test -> no leakage) --------- #
def read_concepts(split):
    df = pd.read_csv(os.path.join(ROCO_DIR, f"{split}_concepts.csv"))
    return {r.ID: [c for c in str(r.CUIs).split(";") if c and c != "nan"]
            for r in df.itertuples()}

cui_train, cui_valid = read_concepts("train"), read_concepts("valid")
freq  = collections.Counter(c for v in cui_train.values() for c in v)
VOCAB = sorted([c for c, n in freq.items() if n >= MIN_FREQ])
IDX   = {c: i for i, c in enumerate(VOCAB)}
C     = len(VOCAB)
print(f"label space: {C} concepts (>= {MIN_FREQ} occurrences)")

def build_records(split, id2cuis, limit=0):
    img_dir = os.path.join(ROCO_DIR, split)
    recs = [{"id": i, "path": os.path.join(img_dir, f"{i}.jpg"),
             "y": [IDX[c] for c in cs if c in IDX]}
            for i, cs in id2cuis.items()
            if os.path.isfile(os.path.join(img_dir, f"{i}.jpg"))]
    recs.sort(key=lambda r: r["id"])
    random.Random(SEED).shuffle(recs)
    return recs[:limit] if limit else recs

train_recs = build_records("train", cui_train, N_TRAIN)
val_recs   = build_records("valid", cui_valid, N_VAL)
print(f"train {len(train_recs)} | valid {len(val_recs)}")
print("concepts per image (train):",
      round(np.mean([len(r["y"]) for r in train_recs]), 2))

In [ ]:
# ------------------------------ Data ------------------------------ #
from torchvision import transforms as T
from torchvision.models import densenet121, DenseNet121_Weights

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]     # ImageNet stats

# NOTE: deliberately NO horizontal flip. Laterality is clinically meaningful, and the
# label space itself contains directionality concepts (AP / PA / sagittal) -- mirroring
# an image would make its own label wrong.
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize(int(IMG_SIZE * 1.14)), T.CenterCrop(IMG_SIZE),
    T.ToTensor(), T.Normalize(MEAN, STD),
])

class RocoCUI(Dataset):
    # n_classes travels with the instance instead of being read from a global,
    # so the dataset still works when it is pickled into a worker process.
    def __init__(self, recs, tf, n_classes):
        self.recs, self.tf, self.C = recs, tf, n_classes
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        try:
            img = Image.open(r["path"]).convert("RGB")   # radiographs are 1-channel
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        y = torch.zeros(self.C); y[r["y"]] = 1.0
        return self.tf(img), y

pin = (device == "cuda")
train_dl = DataLoader(RocoCUI(train_recs, train_tf, C), batch_size=BATCH, shuffle=True,
                      num_workers=WORKERS, pin_memory=pin, drop_last=True)
val_dl   = DataLoader(RocoCUI(val_recs, eval_tf, C), batch_size=BATCH, shuffle=False,
                      num_workers=WORKERS, pin_memory=pin)
print("batches:", len(train_dl), "train /", len(val_dl), "val")

In [ ]:
# ------------------ Model: DenseNet-121 + GeM + linear head ------------------ #
class GeM(nn.Module):
    """Generalised-mean pooling:  (mean(x^p))^(1/p), with p LEARNED.
    p = 1   -> average pooling  (keeps context, dilutes small findings)
    p -> inf-> max pooling      (keeps peak, discards context)
    Learning p lets the network choose its own trade-off."""
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.tensor(p)); self.eps = eps
    def forward(self, x):
        # p is CLAMPED: nothing stops the learned p drifting toward 0, and
        # pow(1/p) then explodes -> NaN weights hours into a run.
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        return F.avg_pool2d(x, x.shape[-2:]).pow(1.0 / p).flatten(1)

class CUINet(nn.Module):
    def __init__(self, n_out):
        super().__init__()
        m = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        self.features = m.features           # ImageNet-pretrained conv stack
        self.pool = GeM()
        self.head = nn.Linear(1024, n_out)   # DenseNet-121 final channels = 1024
    def forward(self, x):
        return self.head(self.pool(F.relu(self.features(x), inplace=True)))

model = CUINet(C).to(device)

# pos_weight = #neg/#pos per class, clamped so ultra-rare concepts don't dominate.
# Without it, ~3 positives vs ~1568 negatives lets the model win by always saying "absent".
cnt  = collections.Counter(i for r in train_recs for i in r["y"])
N    = len(train_recs)
posw = torch.tensor([min((N - max(1, cnt.get(i, 0))) / max(1, cnt.get(i, 0)), POSW_CLAMP)
                     for i in range(C)], dtype=torch.float32, device=device)
print(f"pos_weight: min {posw.min():.1f}  max {posw.max():.1f}")
criterion = nn.BCEWithLogitsLoss(pos_weight=posw)

optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DEC)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)
print("trainable params:", sum(p.numel() for p in model.parameters())/1e6, "M")

In [ ]:
# --------- Metric: samples-average F1 (ImageCLEF's official primary metric) --------- #
def samples_f1(pred_bool, gold_bool):
    """Per-image F1 between predicted and gold concept SETS, averaged over images."""
    tp = (pred_bool & gold_bool).sum(1).astype(np.float64)
    npd = pred_bool.sum(1); ng = gold_bool.sum(1)
    d = npd + ng
    return float(np.where(d > 0, 2 * tp / np.maximum(d, 1), 1.0).mean())

@torch.no_grad()
def collect_scores(dl):
    model.eval(); S, G = [], []
    for x, y in tqdm(dl, desc="scoring", leave=False):
        S.append(torch.sigmoid(model(x.to(device))).float().cpu().numpy())
        G.append(y.numpy().astype(bool))
    return np.concatenate(S), np.concatenate(G)

def tune_global_threshold(S, G):
    best_t, best_f = 0.5, -1.0
    for t in np.arange(0.05, 0.95, 0.01):
        f = samples_f1(S >= t, G)
        if f > best_f: best_t, best_f = float(t), f
    return best_t, best_f

In [ ]:
# ------------------------------ Training loop ------------------------------ #
hist, best_f1, bad, t0 = [], -1.0, 0, time.time()
for ep in range(EPOCHS):
    model.train(); tot = nb = 0
    for x, y in tqdm(train_dl, desc=f"epoch {ep+1}/{EPOCHS}", mininterval=10):
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        optim.zero_grad(set_to_none=True); loss.backward(); optim.step()
        tot += loss.item(); nb += 1
    sched.step()

    S, G = collect_scores(val_dl)
    t_g, f_g = tune_global_threshold(S, G)
    hist.append({"epoch": ep+1, "train_loss": tot/max(1,nb), "val_samples_f1": f_g,
                 "global_tau": t_g, "hours": (time.time()-t0)/3600})
    print(f"[epoch {ep+1}] loss {tot/max(1,nb):.4f} | val samples-F1 {f_g:.4f} (tau={t_g:.2f})")

    torch.save(model.state_dict(), os.path.join(OUT_DIR, "last.pt"))
    if f_g > best_f1:
        best_f1, bad = f_g, 0
        torch.save(model.state_dict(), os.path.join(OUT_DIR, "best.pt"))
        print("  new best -> best.pt")
    else:
        bad += 1; print(f"  no improvement ({bad}/{PATIENCE})")
    json.dump(hist, open(os.path.join(OUT_DIR, "history.json"), "w"), indent=1)
    if bad >= PATIENCE:
        print("[early stop]"); break

In [ ]:
# -------- Per-label thresholds by coordinate ascent (AUEB's winning trick) -------- #
# One global threshold cannot serve both a concept in 20,459 images and one in 10.
# Coordinate ascent: optimise ONE concept's threshold at a time, holding the rest
# fixed, and loop. Incremental -- changing tau[c] only affects images where c flips.
def tune_per_label(S, G, tau0, sweeps=2, n_cand=20):
    n, c_ = S.shape
    tau  = np.full(c_, tau0, dtype=np.float64)
    pred = S >= tau
    tp   = (pred & G).sum(1).astype(np.float64)
    npd  = pred.sum(1).astype(np.float64)
    ng   = G.sum(1).astype(np.float64)
    def score(tp, npd):
        d = npd + ng
        return float(np.where(d > 0, 2*tp/np.maximum(d, 1), 1.0).mean())
    cur = score(tp, npd)
    for s in range(sweeps):
        for c in tqdm(range(c_), desc=f"sweep {s+1}/{sweeps}", leave=False):
            col, g = S[:, c], G[:, c]
            cands  = np.unique(np.quantile(col, np.linspace(0.01, 0.99, n_cand)))
            base   = pred[:, c].copy()
            best_t, best_s = tau[c], cur
            for t in cands:
                newp = col >= t; flip = newp != base
                if not flip.any(): continue
                tp2, np2 = tp.copy(), npd.copy()
                np2[flip] += np.where(newp[flip], 1.0, -1.0)
                tp2[flip] += np.where(newp[flip] & g[flip], 1.0,
                              np.where((~newp[flip]) & base[flip] & g[flip], -1.0, 0.0))
                sc = score(tp2, np2)
                if sc > best_s: best_t, best_s = float(t), sc
            if best_t != tau[c]:
                newp = col >= best_t; flip = newp != base
                npd[flip] += np.where(newp[flip], 1.0, -1.0)
                tp[flip]  += np.where(newp[flip] & g[flip], 1.0,
                             np.where((~newp[flip]) & base[flip] & g[flip], -1.0, 0.0))
                pred[:, c] = newp; tau[c] = best_t; cur = best_s
        print(f"  after sweep {s+1}: samples-F1 = {cur:.4f}")
    return tau, cur

model.load_state_dict(torch.load(os.path.join(OUT_DIR, "best.pt"), map_location=device))
S, G = collect_scores(val_dl)
t_g, f_g = tune_global_threshold(S, G)
print(f"global threshold    : tau={t_g:.2f}  samples-F1 = {f_g:.4f}")
tau, f_pl = tune_per_label(S, G, t_g)
print(f"per-label thresholds: samples-F1 = {f_pl:.4f}   (+{f_pl-f_g:.4f})")

json.dump({"vocab": VOCAB, "min_freq": MIN_FREQ, "img_size": IMG_SIZE,
           "global_tau": t_g, "per_label_tau": tau.tolist(),
           "val_samples_f1_global": f_g, "val_samples_f1_perlabel": f_pl},
          open(os.path.join(OUT_DIR, "thresholds.json"), "w"))
print("\nImageCLEF 2025 reference (test split): AUEB 1st = 0.5888 | 5th = 0.5225")

### What Stage 2 needs from this

`thresholds.json` and `best.pt` are the handoff. Stage 2 (label completion) will:

1. run this model over the **training** set,
2. collect predictions above a **high** confidence bar that are **absent from the labels**,
3. split them into **Type A** (the concept's alias appears non-negated in the caption
   &rarr; provable MedCAT extraction miss, auto-acceptable) and **Type B** (vision-only,
   reported but not auto-accepted).

### Sanity expectations
* Validation samples-F1 should reach roughly **0.50&ndash;0.59** &mdash; the ImageCLEF range.
  Much lower means something is wrong (label space, `pos_weight`, or learning rate).
* Per-label thresholds should add a few points over the single global threshold. If they
  add nothing, check that the score matrix really varies per concept.
* The learned GeM `p` typically settles around 2&ndash;4; inspect with `model.pool.p.item()`.